# 00 — Notebook Principal (Orquestador)
**Disparidades en el Desempeño Saber Pro y su Asociación con el Período de Adopción de IA Generativa: 2021-2024**

**Autor:** Eduardo Andrés Victoria Cadena | Universidad Surcolombiana, 2026

---

Este notebook ejecuta el flujo completo en Google Colab:
1. Instala dependencias y monta Google Drive
2. Prepara datos (DataICFES o simulados)
3. Análisis descriptivo — Parte 1
4. Regresión MCO — Parte 2
5. Diagnósticos
6. Visualizaciones

> **Nota:** Para resultados válidos use los microdatos reales de [DataICFES](https://www.icfes.gov.co/data-icfes/).
> Si no están disponibles, el script genera datos simulados para probar el flujo.

In [ ]:
# ── PASO 0: Instalar paquetes y configurar entorno ────────────────────────
import subprocess, sys

paquetes_extra = ['statsmodels','pyreadstat','openpyxl','linearmodels','pingouin']
for pkg in paquetes_extra:
    try: __import__(pkg)
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('✅ Todos los paquetes listos.')

In [ ]:
# ── PASO 1: Montar Google Drive y configurar rutas ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ⚠ Ajuste RUTA_PROYECTO a la carpeta en su Drive donde subió el repositorio
RUTA_PROYECTO = '/content/drive/MyDrive/IA-Y-EDUCACION-SUPERIOR'

import sys, os
sys.path.insert(0, f'{RUTA_PROYECTO}/python')

for carpeta in ['data/raw','data/processed','outputs/tablas',
                'outputs/figuras','outputs/modelos']:
    os.makedirs(f'{RUTA_PROYECTO}/{carpeta}', exist_ok=True)

print(f'Proyecto en: {RUTA_PROYECTO}')
print('Estructura de carpetas creada.')

In [ ]:
# ── PASO 2: Preparación de datos ─────────────────────────────────────────
print('='*70)
print('PASO 2: Preparación de datos')
print('='*70)

import numpy as np, pandas as pd, glob, re, warnings
warnings.filterwarnings('ignore')
from utils import (IES_MUESTRA, PROGRAMAS_MUESTRA, ANOS_PREVIO, ANOS_IA,
                   DISTANCIAS_BOGOTA, MODULOS_GENERICOS,
                   calcular_puntaje_generico, guardar_tabla)

PROC_DIR = f'{RUTA_PROYECTO}/data/processed'
RAW_DIR  = f'{RUTA_PROYECTO}/data/raw'

# Verificar si hay datos reales
archivos_raw = (glob.glob(f'{RAW_DIR}/saberpro_*.csv')
              + glob.glob(f'{RAW_DIR}/saberpro_*.sav')
              + glob.glob(f'{RAW_DIR}/saberpro_*.xlsx'))

if archivos_raw:
    print(f'Datos reales encontrados: {archivos_raw}')
    print('Ejecute la celda del notebook 01_preparar_datos.ipynb')
    # O importe la función directamente:
    # %run {RUTA_PROYECTO}/python/01_preparar_datos.ipynb
else:
    print('⚠ No se encontraron datos en data/raw/')
    print('Generando datos SIMULADOS para probar el flujo...')
    # Función generadora (copia compacta)
    from utils import DISTANCIAS_BOGOTA, MODULOS_GENERICOS, calcular_puntaje_generico

    def generar_simulados(n=3000, semilla=42):
        rng   = np.random.default_rng(semilla)
        deptos = list(DISTANCIAS_BOGOTA.keys())
        prob_d = [0.30,0.20,0.20,0.12,0.10,0.08]
        ies_map = {
            'BOGOTA'   :['UNIVERSIDAD NACIONAL DE COLOMBIA',
                         'UNIVERSIDAD DISTRITAL FRANCISCO JOSE DE CALDAS'],
            'ANTIOQUIA':['UNIVERSIDAD DE ANTIOQUIA'],
            'VALLE'    :['UNIVERSIDAD DEL VALLE'],
            'HUILA'    :['UNIVERSIDAD SURCOLOMBIANA'],
            'NARINO'   :['UNIVERSIDAD DE NARINO'],
            'TOLIMA'   :['UNIVERSIDAD DEL TOLIMA'],
        }
        depto_arr = rng.choice(deptos, n, p=prob_d)
        anio_arr  = rng.choice([2021,2022,2023,2024], n)
        df = pd.DataFrame({
            'id_estudiante'   : range(n),
            'ANIO_APLICACION' : anio_arr,
            'depto_ies'       : depto_arr,
            'nombre_ies'      : [rng.choice(ies_map[d]) for d in depto_arr],
            'programa_academico': rng.choice(
                ['ECONOMIA','ADMINISTRACION DE EMPRESAS','CONTADURIA PUBLICA'],
                n, p=[0.25,0.45,0.30]),
            'periodo_ia'      : pd.Series(anio_arr).isin(ANOS_IA).astype(int).values,
            'estrato'         : rng.choice(range(1,7),n,p=[0.10,0.25,0.30,0.20,0.10,0.05]),
            'genero'          : rng.binomial(1,0.48,n),
            'nivel_educ_padre': rng.choice(range(1,8),n,p=[0.05,0.10,0.15,0.30,0.18,0.15,0.07]),
            'estu_trabaja'    : rng.binomial(1,0.40,n),
            'estu_cabeza_familia': rng.binomial(1,0.15,n),
            'internet'        : rng.binomial(1,0.82,n),
            'area_residencia' : rng.binomial(1,0.78,n),
            'naturaleza_ies'  : np.zeros(n,dtype=int),
            'puntaje_saber11' : rng.normal(280,45,n),
        })
        df['distancia_bogota_km'] = df['depto_ies'].map(DISTANCIAS_BOGOTA)
        for d in ['antioquia','valle','huila','narino','tolima']:
            df[f'd_{d}'] = (df['depto_ies']==d.upper()).astype(int)
        efecto_ia    = -2.5 * df['periodo_ia']
        efecto_depto = df['depto_ies'].map({'BOGOTA':0,'ANTIOQUIA':-4,'VALLE':-5,
                                             'HUILA':-9,'NARINO':-14,'TOLIMA':-7})
        base = 145 + efecto_ia + 3*df['estrato'] + efecto_depto + 0.05*(df['puntaje_saber11']-280)
        for i,m in enumerate(MODULOS_GENERICOS):
            off = [0,-5,2,1,-8][i]
            df[m] = np.clip(base + off + rng.normal(0,12,n), 0, 300)
        return calcular_puntaje_generico(df)

    df = generar_simulados()
    df.to_pickle(f'{PROC_DIR}/datos_analisis.pkl')
    df.to_csv(f'{PROC_DIR}/datos_analisis.csv', index=False)

print(f'\n✅ Muestra: {len(df):,} filas | {df.shape[1]} columnas')
print(f'  pre-IA (0): n={(df.periodo_ia==0).sum():,}')
print(f'  IA     (1): n={(df.periodo_ia==1).sum():,}')
df.head(3)

In [ ]:
# ── PASO 3: Análisis Descriptivo (Parte 1) ───────────────────────────────
print('='*70)
print('PASO 3: Análisis Descriptivo — Parte 1')
print('='*70)

from scipy import stats
from statsmodels.stats.multitest import multipletests
from utils import ETIQUETAS_VARS, CONTROLES, DUMMIES_DEPTO, ALPHA

TBL_DIR = f'{RUTA_PROYECTO}/outputs/tablas'

def comparar_periodos(df):
    rows = []
    d0, d1 = df[df['periodo_ia']==0], df[df['periodo_ia']==1]
    vars_c = ['puntaje_saberpro_generico'] + MODULOS_GENERICOS + ['puntaje_saber11','estrato']
    for v in [x for x in vars_c if x in df.columns]:
        x0, x1 = d0[v].dropna(), d1[v].dropna()
        tt = stats.ttest_ind(x0, x1, equal_var=False)
        mw = stats.mannwhitneyu(x0, x1, alternative='two-sided')
        se_d = np.sqrt(x0.var()/len(x0) + x1.var()/len(x1))
        rows.append({'variable':v,'tipo':'continua',
                     'media_p0':round(x0.mean(),2),'de_p0':round(x0.std(),2),'n_p0':len(x0),
                     'media_p1':round(x1.mean(),2),'de_p1':round(x1.std(),2),'n_p1':len(x1),
                     'delta':round(x1.mean()-x0.mean(),2),
                     't_stat':round(tt.statistic,3),'p_ttest':round(tt.pvalue,4),
                     'p_mw':round(mw.pvalue,4)})
    for v in ['genero','estu_trabaja','internet','area_residencia','naturaleza_ies']:
        if v not in df.columns: continue
        x0, x1 = d0[v].dropna(), d1[v].dropna()
        tab = pd.crosstab(df[v], df['periodo_ia'])
        chi2, p_chi, _, _ = stats.chi2_contingency(tab, correction=True)
        rows.append({'variable':v,'tipo':'dicotomica',
                     'media_p0':round(x0.mean()*100,1),'de_p0':np.nan,'n_p0':len(x0),
                     'media_p1':round(x1.mean()*100,1),'de_p1':np.nan,'n_p1':len(x1),
                     'delta':round((x1.mean()-x0.mean())*100,1),
                     't_stat':round(chi2,3),'p_ttest':round(p_chi,4),'p_mw':np.nan})
    tbl = pd.DataFrame(rows)
    tbl['etiqueta'] = tbl['variable'].map(ETIQUETAS_VARS).fillna(tbl['variable'])
    pvals = tbl['p_ttest'].dropna().values
    _, p_holm,_,_ = multipletests(pvals, method='holm')
    _, p_bh,_,_   = multipletests(pvals, method='fdr_bh')
    tbl.loc[tbl['p_ttest'].notna(),'p_holm'] = p_holm
    tbl.loc[tbl['p_ttest'].notna(),'p_bh']   = p_bh
    return tbl

tabla3 = comparar_periodos(df)
guardar_tabla(tabla3, 'T3_descriptivo_periodos', TBL_DIR)
print('Tabla 3 guardada.')
print(tabla3[['etiqueta','media_p0','media_p1','delta','p_ttest']].to_string(index=False))

In [ ]:
# ── PASO 4: Regresión MCO (Parte 2) ──────────────────────────────────────
print('='*70)
print('PASO 4: Regresión Lineal Múltiple MCO — Parte 2')
print('='*70)

import statsmodels.formula.api as smf

vars_dep = ['puntaje_saberpro_generico'] + [m for m in MODULOS_GENERICOS if m in df.columns]
specs    = {'base':None,'ef_ies':'ies','ef_mun':'mun'}

def construir_formula(var_dep, ef=None):
    rhs = ['periodo_ia']
    rhs += [d for d in DUMMIES_DEPTO if d in df.columns]
    if 'distancia_bogota_km' in df.columns: rhs.append('distancia_bogota_km')
    rhs += [c for c in CONTROLES if c in df.columns]
    if ef == 'ies' and 'nombre_ies' in df.columns:   rhs.append('C(nombre_ies)')
    elif ef == 'mun' and 'area_residencia' in df.columns: rhs.append('C(area_residencia)')
    return f'{var_dep} ~ ' + ' + '.join(rhs)

tabla4_rows = []
for vd in vars_dep:
    for sp_name, ef_val in specs.items():
        formula = construir_formula(vd, ef=ef_val)
        cols = [vd]+[c for c in CONTROLES+DUMMIES_DEPTO+['periodo_ia','distancia_bogota_km']
                     if c in df.columns]
        df_c = df[list(set(cols)&set(df.columns))].dropna()
        if len(df_c)<30: continue
        mod   = smf.ols(formula, data=df_c).fit()
        if 'nombre_ies' in df_c.columns:
            grp = df_c['nombre_ies'].astype('category').cat.codes
            try:  mod_use = mod.get_robustcov_results(cov_type='cluster',groups=grp)
            except: mod_use = mod.get_robustcov_results(cov_type='HC3')
        else:
            mod_use = mod.get_robustcov_results(cov_type='HC3')
        for var, coef, se, t, p in zip(mod_use.params.index, mod_use.params,
                                        mod_use.bse, mod_use.tvalues, mod_use.pvalues):
            tabla4_rows.append({'var_dep':vd,'spec':sp_name,'variable':var,
                                 'coef':round(coef,4),'se':round(se,4),
                                 't_stat':round(t,4),'p_valor':round(p,4),
                                 'n':len(df_c),'r2_adj':round(mod.rsquared_adj,4)})
        print(f'  {vd} | {sp_name}: n={len(df_c):,} | R²adj={mod.rsquared_adj:.3f}')

tabla4 = pd.DataFrame(tabla4_rows)
tabla4['sig'] = tabla4['p_valor'].apply(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '.' if p<0.10 else '')
guardar_tabla(tabla4, 'T4_coeficientes_todos_modelos', TBL_DIR)
print('\n✅ Tabla 4 guardada.')

# Resumen β_IA
beta_ia = tabla4[tabla4['variable']=='periodo_ia'][['var_dep','spec','coef','se','t_stat','p_valor','sig','n','r2_adj']].copy()
beta_ia['modulo'] = beta_ia['var_dep'].map(ETIQUETAS_VARS).fillna(beta_ia['var_dep'])
guardar_tabla(beta_ia, 'T4b_beta_IA_por_modulo', TBL_DIR)
print('\n=== β_IA por módulo y especificación ===')
print(beta_ia[['modulo','spec','coef','se','p_valor','sig']].to_string(index=False))

In [ ]:
# ── PASO 5: Diagnósticos ──────────────────────────────────────────────────
print('='*70)
print('PASO 5: Diagnósticos')
print('='*70)

from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

diag_rows = []
for vd in vars_dep:
    formula = construir_formula(vd, ef=None)
    cols = [vd]+[c for c in CONTROLES+DUMMIES_DEPTO+['periodo_ia','distancia_bogota_km']
                 if c in df.columns]
    df_c = df[list(set(cols)&set(df.columns))].dropna()
    if len(df_c)<30: continue
    mod = smf.ols(formula, data=df_c).fit()
    res = mod.resid

    # Normalidad
    if len(res)<5000: sw_s, sw_p = stats.shapiro(res); prueba_n='Shapiro-Wilk'
    else:             sw_s, sw_p = stats.kstest(stats.zscore(res),'norm'); prueba_n='KS'
    # Breusch-Pagan
    bp_stat, bp_p, _, _ = het_breuschpagan(res, mod.model.exog)
    # Durbin-Watson
    dw = durbin_watson(res)
    # VIF máximo
    X = mod.model.exog; cols_x = mod.model.exog_names
    vifs = [variance_inflation_factor(X,i) for i in range(len(cols_x))]
    max_vif = max(vifs)
    # RESET
    try: rst = linear_reset(mod,power=3,use_f=True); rst_p = rst.pvalue
    except: rst_p = np.nan

    diag_rows.append({'modelo':vd,'prueba_norm':prueba_n,
                      'p_norm':round(sw_p,4),'p_bp':round(bp_p,4),
                      'max_vif':round(max_vif,3),'DW':round(dw,4),'p_reset':round(rst_p,4)})
    print(f'  {vd}: norm_p={sw_p:.3f} | BP_p={bp_p:.3f} | maxVIF={max_vif:.1f} | DW={dw:.3f}')

tbl_diag = pd.DataFrame(diag_rows)
guardar_tabla(tbl_diag, 'T_diagnosticos_resumen', TBL_DIR)
print('\n✅ Diagnósticos guardados.')

In [ ]:
# ── PASO 6: Visualizaciones ───────────────────────────────────────────────
print('='*70)
print('PASO 6: Visualizaciones')
print('='*70)

import matplotlib.pyplot as plt
import seaborn as sns
from utils import COLORES_DEPTO

FIG_DIR = f'{RUTA_PROYECTO}/outputs/figuras'

# Fig 2 rápida (histograma) como demo en el main notebook
fig, ax = plt.subplots(figsize=(10, 5))
for per, lbl, color in [(0,'2021-2022','#1565C0'),(1,'2023-2024','#B71C1C')]:
    datos = df[df['periodo_ia']==per]['puntaje_saberpro_generico'].dropna()
    ax.hist(datos, bins=35, density=True, alpha=0.45, color=color, label=lbl)
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(datos)
    x   = np.linspace(datos.min(), datos.max(), 300)
    ax.plot(x, kde(x), color=color, linewidth=2)
    ax.axvline(datos.mean(), color=color, linestyle='--', linewidth=1.2)
ax.set(title='Figura 2. Distribución del puntaje genérico por período',
       xlabel='Puntaje genérico', ylabel='Densidad')
ax.legend(title='Período')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/F2_histograma_periodos.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nPara las 7 figuras completas ejecute: 05_visualizaciones.ipynb')
print('✅ Análisis completado.')

In [ ]:
# ── RESUMEN FINAL ────────────────────────────────────────────────────────
print('='*70)
print('RESUMEN DE RESULTADOS CLAVE')
print('='*70)

print('\n1. β_IA (período de IA generativa) por módulo y especificación:')
print(beta_ia[['modulo','spec','coef','se','p_valor','sig']].to_string(index=False))

print('\n2. Diagnósticos del modelo base:')
print(tbl_diag[['modelo','p_norm','p_bp','max_vif','DW','p_reset']].to_string(index=False))

print('\n3. Outputs generados:')
for f in sorted(os.listdir(TBL_DIR)):
    print(f'   tablas/{f}')
for f in sorted(os.listdir(FIG_DIR)):
    print(f'   figuras/{f}')

print('''
INTERPRETACIÓN (condicional, NO causal):
  * β_IA > 0 y sig.: asociación positiva con el período de IA.
  * β_IA < 0 y sig.: asociación negativa con el período de IA.
  * Coeficiente departamental < 0: brecha condicional negativa vs Bogotá D.C.
  * distancia_bogota_km < 0 y sig.: centralización educativa.
''')